## BONUS: Live Computer Vision with YOLO

Now let's see machine learning in action! We'll use a lightweight pre-trained YOLOv8n (You Only Look Once) model to detect objects in real time using your webcam.

**What is YOLO?**
- YOLO is a real-time object detection system
- It can identify and locate multiple objects in images/video
- Used in self-driving cars, security systems, and mobile apps
- Demonstrates how modern computer vision can detect multiple objects in real time

**Note:** This is a first look at pre-trained model inference. We'll explore the ideas behind computer vision models later in the course. For now, just enjoy the demonstration!

In [1]:
# Import required libraries for computer vision
import cv2
from ultralytics import YOLO
import time
import threading

# Check if the webcam is available without leaving it open
def check_webcam(timeout=5):
    """Check webcam availability, with a timeout to avoid hanging."""
    result = [False]
    
    def try_open():
        cap = cv2.VideoCapture(0)
        try:
            result[0] = cap.isOpened()
        finally:
            cap.release()
    
    thread = threading.Thread(target=try_open, daemon=True)
    thread.start()
    thread.join(timeout=timeout)
    
    if thread.is_alive():
        print(f"✗ Webcam check timed out after {timeout} seconds. Will use sample image instead.")
        return False
    
    if result[0]:
        print("✓ Webcam detected and ready!")
        return True
    else:
        print("✗ No webcam detected (or timed out). Will use sample image instead.")
        return False

# Test webcam availability
webcam_available = check_webcam()

print("\nSetting up YOLO object detection...")
print("This may take a moment to download the model weights...")

# Load pre-trained YOLO model
try:
    model_yolo = YOLO('yolov8n.pt')
    print("✓ YOLO model loaded successfully!")
    print(f"Model: YOLOv8 Nano")
    print(f"Classes: {len(model_yolo.names)} object types can be detected")
    
    print("Example detectable objects:")
    example_classes = ['person', 'car', 'dog', 'cat', 'bottle', 'chair', 'laptop', 'cell phone']
    for cls in example_classes:
        if cls in model_yolo.names.values():
            print(f"  - {cls.title()}")
    
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    print("Please ensure you have internet connection for model download.")
    model_yolo = None

✓ Webcam detected and ready!

Setting up YOLO object detection...
This may take a moment to download the model weights...
✓ YOLO model loaded successfully!
Model: YOLOv8 Nano
Classes: 80 object types can be detected
Example detectable objects:
  - Person
  - Car
  - Dog
  - Cat
  - Bottle
  - Chair
  - Laptop
  - Cell Phone


In [2]:
import matplotlib.pyplot as plt

# Live webcam object detection function
def run_live_detection(duration=5 * 60):
    """
    Run live object detection using webcam
    
    Args:
        duration: Maximum detection time in seconds
    """
    if not webcam_available or model_yolo is None:
        print("Webcam or YOLO model not available.")
        return
    
    print(f"Starting live object detection for up to {duration // 60} minutes...")
    print("Press 'q' or Esc, or close the window, to stop. Press 'c' to capture a frame.")
    print("Objects will be detected and labeled in real-time!")
    
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Could not open the webcam.")
        cap.release()
        return
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    window_name = 'YOLO Live Object Detection'
    
    start_time = time.time()
    frame_count = 0
    
    try:
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        
        while True:
            # After the first frame, stop if the user closes the display window
            if frame_count > 0:
                try:
                    if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                        print("Detection window closed by user")
                        break
                except cv2.error:
                    print("Detection window closed by user")
                    break
            
            ret, frame = cap.read()
            if not ret:
                print("Failed to capture frame")
                break
            
            # Run YOLO detection
            results = model_yolo(frame, verbose=False)
            annotated_frame = results[0].plot()
            
            # Add performance info
            frame_count += 1
            fps = frame_count / (time.time() - start_time)
            cv2.putText(annotated_frame, f'FPS: {fps:.1f}', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow(window_name, annotated_frame)
            
            # Check for key presses
            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), 27):
                print("Quit requested by user")
                break
            elif key == ord('c'):
                filename = f'captured_frame_{int(time.time())}.jpg'
                cv2.imwrite(filename, annotated_frame)
                print(f"Frame captured: {filename}")
            
            # Check time limit
            if time.time() - start_time >= duration:
                print(f"Time limit ({duration // 60} minutes) reached")
                break
                
    except KeyboardInterrupt:
        print("Detection stopped by user")
    finally:
        cap.release()
        
        # On macOS, explicitly destroy the named window and process the
        # pending GUI events so the frozen final frame disappears.
        try:
            cv2.destroyWindow(window_name)
        except cv2.error:
            pass
        cv2.destroyAllWindows()
        for _ in range(5):
            try:
                cv2.waitKey(1)
            except cv2.error:
                break
            time.sleep(0.01)
        
        total_time = time.time() - start_time
        avg_fps = frame_count / total_time if total_time > 0 else 0
        print(f"\nDetection Summary:")
        print(f"  Total time: {total_time:.1f} seconds")
        print(f"  Frames processed: {frame_count}")
        print(f"  Average FPS: {avg_fps:.1f}")


# Fallback: detect objects in a sample image
def run_image_detection():
    """Run object detection on a sample image"""
    if model_yolo is None:
        print("YOLO model not available.")
        return
        
    print("Running YOLO detection on sample image...")
    
    try:
        import urllib.request
        sample_image_url = "https://ultralytics.com/images/bus.jpg"
        urllib.request.urlretrieve(sample_image_url, "sample_image.jpg")
        
        results = model_yolo("sample_image.jpg")
        annotated_image = results[0].plot()
        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_image_rgb)
        plt.axis('off')
        plt.title('YOLO Object Detection Results')
        plt.tight_layout()
        plt.show()
        
        print("\nDetection Results:")
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                for box in boxes:
                    class_id = int(box.cls[0])
                    confidence = float(box.conf[0])
                    class_name = model_yolo.names[class_id]
                    print(f"  Detected: {class_name} (confidence: {confidence:.2f})")
        
    except Exception as e:
        print(f"Error in image detection: {e}")
        
print("YOLO functions defined successfully!")

YOLO functions defined successfully!


In [3]:
# YOLO Object Detection Demo
print("YOLO Object Detection Demo")
print("=" * 50)

if model_yolo is None:
    print("Cannot run YOLO demo — model failed to load.")
elif webcam_available:
    print("Webcam detected! Starting live detection for up to 5 minutes...")
    print("Press 'q' or Esc, or close the window, to stop. Press 'c' to capture a frame.\n")
    run_live_detection(duration=5 * 60)
else:
    print("No webcam available — running sample image detection...\n")
    run_image_detection()

print("\nYOLO Demo Complete!")
print("You just experienced real-time object detection with a pre-trained model!")
print("Later in the course, we'll explore how computer vision models make these predictions.")

YOLO Object Detection Demo
Webcam detected! Starting live detection for up to 5 minutes...
Press 'q' or Esc, or close the window, to stop. Press 'c' to capture a frame.

Starting live object detection for up to 5 minutes...
Press 'q' or Esc, or close the window, to stop. Press 'c' to capture a frame.
Objects will be detected and labeled in real-time!
Quit requested by user

Detection Summary:
  Total time: 141.6 seconds
  Frames processed: 1570
  Average FPS: 11.1

YOLO Demo Complete!
You just experienced real-time object detection with a pre-trained model!
Later in the course, we'll explore how computer vision models make these predictions.
